### Fine-tuning a LLaMA 3 8B model for Multifaceted Analysis of Subjective Text (MAST)

This notebook finetunes and evaluates a LLaMA 3 8B model for MAST tasks, namely offensive speech detection. 1 indicates the presence of offensive speech and 0 indicates its absence. The ORCA 2 inspired fine-tuning method is used to adapt the model.

- A synthetically generated dataset was used for fine-tuning and evaluating the model. For this dataset, the fine-tuning and evaluation pipeline was run 3 times using a different train-test split each time, namely with seed 42, 55 and 777. The train and test splits can be found in the dataset folder contained within this directory.
- The average score of each metric across the 3 runs was computed and the results are listed in the final report
- An additional humanised evaluation dataset is used to evaluate the fine-tuned model's performance on feedback that more closely mirrors student feedback text.

This pipeline is executed on google colab using a T4 GPU. To run, select and upload datasets from the `dataset` folder and this notebook, `orca2_finetuning_mast`, to google colab. 

In [ ]:
!pip uninstall -y unsloth unsloth_zoo
!pip -q install -U pip setuptools wheel
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 79.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 127.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 53.4 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 90.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 89.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 51.2 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

# load in pre-trained model and tokeniser
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [ ]:
# converts base model to a LoRA fine-tunable model
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.4.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
# format data into alpaca style format for instruction tuning
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }


Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
from datasets import load_dataset, Dataset
import random
from collections import Counter

raw_dataset = load_dataset(
    "json",
    data_files="/content/train_55.jsonl",
    split="train",
)

In [ ]:
train_raw = raw_dataset
train_dataset = train_raw.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = -1,
        num_train_epochs=4,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/700 [00:00<?, ? examples/s]

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 700 | Num Epochs = 4 | Total steps = 352
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.144545
20,0.693741
30,0.534036
40,0.484287
50,0.461570
60,0.445819
70,0.435332
80,0.420402
90,0.447974
100,0.362904


In [ ]:
# load test data
import json

test_dataset_path = "test_55.jsonl"
rows = []
with open(test_dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))
test_dataset = rows

In [ ]:
from unsloth import FastLanguageModel
from tqdm import tqdm
import torch

FastLanguageModel.for_inference(model)
model.eval()
preds = []
labels = []

# tqdm enables the progress tracking of
# the inference process
for example in tqdm(test_dataset):
    instruction = example["instruction"]
    sentence    = example["input"]
    gold_label  = example["output"]
    prompt = alpaca_prompt.format(instruction, sentence,"",)

    # generate model inference based on formatted input
    inputs = tokenizer([prompt],return_tensors="pt",).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=16,
                                 use_cache=False, do_sample=False,)

    # process output
    decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    if "### Response:" in decoded_output :
        answer_part = decoded_output.split("### Response:")[-1].strip()
    else:
        answer_part = decoded_output.strip()

    if "0" in answer_part:
        pred_label = 0
    else:
        pred_label = 1

    preds.append(pred_label)
    labels.append(gold_label)

100%|██████████| 300/300 [21:32<00:00,  4.31s/it]


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# compute accuracy and macro averaged metrics
accuracy = accuracy_score(labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support( labels, preds, average="macro",)

print(f"accuracy:{accuracy}, precision:{precision}, recall:{recall}, f1 score:{f1}")

Accuracy : 0.9066666666666666
Precision: 0.9008928571428572
Recall   : 0.9207271364317842
F1 score : 0.9046148255813953
